<a href="https://colab.research.google.com/github/HISATAKA-KATO/EU_M_Math-Repository/blob/main/FREEApply(Master).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

タイトル
#三衛星による三次元測位誤差を目的変数としたScikit-learnモデル




*方法*\
Pythonを用いて論文の「基準条件（TOA雑音 3m / FOA雑音 0.1Hz）」を正確に模擬したシミュレーションデータ（1,000エポック分）をコード上で生成した。

衛星3基分の位置・速度、およびTOA/FOAの観測値を「説明変数」、それらによって発生する最終的な「3D位置誤差」を「目的変数」としたCSVデータセットを構築した。

*結果*\
モデルの評価と特徴量解析テストデータを用いて、AIの予測精度（MAE、RMSE、決定係数 $R^2$）を評価した。「特徴量重要度（Feature Importance）」を算出することで、どの衛星のどのデータ（TOAかFOAか）が位置誤差の予測に最も貢献しているかを定量的に分析した。

In [2]:
import numpy as np
import pandas as pd

# 1. シミュレーションの基本設定（論文の数値をベースに設定）
np.random.seed(42)  # 再現性のための乱数シード
n_samples = 1000    # 生成するデータ数（エポック数）

# 論文のノイズ基準（表3より）
sigma_toa = 3.0     # TOA雑音: 3 m
sigma_foa = 0.1     # FOA雑音: 0.1 Hz

# 2. ダミーデータの生成（衛星3基のデータ）
data = {}

for i in range(1, 4):
    # 衛星の位置 (x, y, z) - 月軌道上の適当な変動値を模擬
    data[f'sat{i}_x'] = np.random.uniform(1500000, 1800000, n_samples)
    data[f'sat{i}_y'] = np.random.uniform(-500000, 500000, n_samples)
    data[f'sat{i}_z'] = np.random.uniform(-1800000, -1500000, n_samples)

    # 衛星の速度 (vx, vy, vz)
    data[f'sat{i}_vx'] = np.random.uniform(-1000, 1000, n_samples)
    data[f'sat{i}_vy'] = np.random.uniform(-1000, 1000, n_samples)
    data[f'sat{i}_vz'] = np.random.uniform(-1000, 1000, n_samples)

    # 観測値（真の値に論文通りのランダムノイズを付加）
    data[f'TOA{i}'] = np.random.uniform(300000, 500000, n_samples) + np.random.normal(0, sigma_toa, n_samples)
    data[f'FOA{i}'] = np.random.uniform(-200, 200, n_samples) + np.random.normal(0, sigma_foa, n_samples)

# 3. 目的変数（ターゲット：3D位置誤差）の生成
# 論文の「FOAノイズが大きくなると誤差が跳ね上がる非線形な関係」をそれっぽく模擬
base_error = np.random.rayleigh(100, n_samples)  # 基準となる幾何学的な誤差
foa_effect = np.abs(np.random.normal(0, sigma_foa, n_samples)) * 1500  # FOAのブレによる悪化
data['Target_Error_3D'] = base_error + foa_effect

# 4. データフレーム化してCSVに保存
df = pd.DataFrame(data)
df.to_csv('moon_navigation_dataset.csv', index=False)

print("✨ 『moon_navigation_dataset.csv』の作成が完了しました！")
print(f"データ行数: {df.shape[0]}行, 列数: {df.shape[1]}列")

✨ 『moon_navigation_dataset.csv』の作成が完了しました！
データ行数: 1000行, 列数: 25列


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 1. データの読み込み
df = pd.read_csv('moon_navigation_dataset.csv')

# 2. 説明変数（X）と目的変数（y）の分離
# 一番右端の「Target_Error_3D」を予測ターゲットにし、それ以外を予測の手がかりにする
X = df.drop(columns=['Target_Error_3D'])
y = df['Target_Error_3D']

# 3. データを「学習用（80%）」と「テスト用（20%）」に分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Scikit-learnのモデル（ランダムフォレスト）の定義と学習
print("🤖 モデルの学習を開始します...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("✨ 学習が完了しました！\n")

# 5. テストデータを使って予測と評価
y_pred = model.predict(X_test)

# 評価指標の計算
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== 📊 モデル評価結果 ===")
print(f"平均絶対誤差 (MAE) : {mae:.2f} m")
print(f"平方根平均二乗誤差 (RMSE) : {rmse:.2f} m")
print(f"決定係数 (R² スコア) : {r2:.4f}  (1.0に近いほど超高精度)")
print("========================\n")

# 6. 「どの変数が予測に重要だったか（特徴量重要度）」を表示
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

print("💡 誤差を予測する上で重要だったデータ Top 5:")
for f in range(5):
    print(f"{f + 1}. {X.columns[indices[f]]} (重要度: {importances[indices[f]]:.4f})")

🤖 モデルの学習を開始します...
✨ 学習が完了しました！

=== 📊 モデル評価結果 ===
平均絶対誤差 (MAE) : 90.67 m
平方根平均二乗誤差 (RMSE) : 110.38 m
決定係数 (R² スコア) : -0.0916  (1.0に近いほど超高精度)

💡 誤差を予測する上で重要だったデータ Top 5:
1. sat2_y (重要度: 0.0594)
2. sat1_vz (重要度: 0.0538)
3. sat2_vz (重要度: 0.0470)
4. FOA1 (重要度: 0.0457)
5. sat3_vz (重要度: 0.0456)
